# Automatic Image Colorization (SVR + MRF/ICM)

Implementation of the approach described in *"Automatic Image Colorization"* (Ho & Ramesh, CS229).

**Pipeline**

1. Convert each image to **YUV** (Y = luminance, U/V = chrominance). The model only sees **Y** and predicts **U, V**.
2. Segment the image into **SLIC superpixels**.
3. For every superpixel take a **10 x 10 patch around its centroid** and compute its **2D FFT** to get a texture feature vector.
4. Train **two Support Vector Regressors** (one for U, one for V) mapping features -> mean U / mean V of the superpixel.
5. At test time, run the SVRs to get an initial colour per superpixel, then model each channel as a **Markov Random Field** over the superpixels and minimise its energy with **Iterated Conditional Modes (ICM)** to smooth neighbouring segments.
6. Recombine the original Y with the predicted U, V and convert back to RGB.

---

## Where to put your dataset

Create a folder called **`dataset`** in the **same directory as this notebook** and put your colour photos in it.
Two layouts are supported:

**Option A (simplest)** - all images directly inside `dataset/`; the notebook splits them randomly into train / test:
```
your_project_folder/
|-- Automatic_Image_Colorization.ipynb
`-- dataset/
    |-- img001.jpg
    |-- img002.jpg
    `-- ...
```

**Option B** - you already have a split:
```
your_project_folder/
|-- Automatic_Image_Colorization.ipynb
`-- dataset/
    |-- train/   <- training images
    `-- test/    <- test images
```

* Use **colour** photos (JPG / PNG / ...). Images that are already grayscale are skipped automatically.
* The paper used ~216 Flickr photos tagged `yellowstone` + `landscape` (98 train / 118 test), but any single *theme* (beaches, forests, mountains...) works best, because the model is trained for one image type.
* Optional: put grayscale (or colour) images you want to colorize in an **`input_images/`** folder; the last section colorizes them.
* If you run on **Google Colab**, upload the folder or mount Drive and change `DATA_DIR` in the config cell below.

## 1. Setup

In [ ]:
# Uncomment if packages are missing (e.g. on Colab / a fresh environment)
# !pip install -q numpy scipy scikit-image scikit-learn matplotlib pillow joblib tqdm

In [ ]:
import random
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage
from skimage import color
from skimage.segmentation import slic, mark_boundaries, relabel_sequential
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import joblib

try:
    from tqdm.auto import tqdm
except ImportError:                       # tqdm is optional
    def tqdm(x, **kwargs):
        return x

%matplotlib inline

## 2. Configuration

All tunable settings live in this one cell.

In [ ]:
# ----------------------------- Paths -----------------------------
DATA_DIR   = Path("dataset")        # <-- PUT YOUR DATASET HERE (see instructions above)
INPUT_DIR  = Path("input_images")   # optional: extra images you want to colorize at the end
OUTPUT_DIR = Path("outputs")        # results (figures, colorized images) are written here
MODEL_DIR  = Path("models")         # trained model is saved here

# --------------------------- Data / split ------------------------
IMG_WIDTH      = 500                # all images are rescaled to this width (as in the paper)
TRAIN_FRACTION = 0.8                # only used for "Option A" (flat dataset/ folder). Paper used 98 train / 118 test.
GRAY_TOL       = 0.01               # images whose channels differ by less than this are treated as already-grayscale
SEED           = 42

# ------------------------ Segmentation / features ----------------
N_SEGMENTS  = 250                   # approx. number of SLIC superpixels per image
COMPACTNESS = 0.1                   # SLIC compactness (suited to a grayscale image in [0, 1])
PATCH       = 10                    # side of the square patch (10 x 10) around each superpixel centroid

# ------------------------------- SVR ------------------------------
# Paper values: epsilon = 0.0625, C = 0.125. If RUN_TUNING is False these are used for both channels.
SVR_C, SVR_EPSILON = 0.125, 0.0625
RUN_TUNING         = True           # search C / epsilon / (gamma, sigma) on a validation split (recommended)
VAL_FRACTION       = 0.15           # share of the TRAINING images held out for tuning
C_GRID             = [0.125, 1.0, 8.0]
EPS_GRID           = [0.002, 0.01, 0.03, 0.0625]
TUNE_SUBSAMPLE     = 6000           # superpixels used per SVR fit during the grid search (keeps it fast)
MAX_TRAIN_SAMPLES  = 40000          # cap on superpixels for the final SVR fit (None = use all). Lower = faster.

# ------------------------------- MRF / ICM ------------------------
ICM_GAMMA, ICM_SIGMA = 2.0, 0.1     # paper: gamma = 2. sigma is not given in the paper (U/V live in ~[-0.6, 0.6])
GAMMA_GRID = [0.5, 2.0, 8.0, 32.0]
SIGMA_GRID = [0.05, 0.1, 0.2]
NEIGHBOR_PERCENTILE = 75            # keep neighbour edges whose feature distance is below this percentile (per image)
ICM_MAX_ITER, ICM_TOL = 100, 1e-5

# ------------------------------ Display ---------------------------
N_SHOW = 6                          # number of test images in the results gallery

random.seed(SEED)
np.random.seed(SEED)
for d in (OUTPUT_DIR, MODEL_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 3. Load the dataset

Images are loaded, rescaled to a constant width, and already-grayscale images are removed.

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


def list_images(folder):
    return sorted(p for p in Path(folder).rglob("*") if p.suffix.lower() in IMG_EXT)


def load_rgb(path, width=IMG_WIDTH):
    """Load any image as an RGB float array in [0, 1] with a fixed width."""
    img = Image.open(path).convert("RGB")
    w, h = img.size
    new_h = max(1, round(h * width / w))
    img = img.resize((width, new_h), Image.LANCZOS)
    return np.asarray(img, dtype=np.float64) / 255.0


def is_grayscale(rgb, tol=GRAY_TOL):
    return np.abs(rgb[..., 0] - rgb[..., 1]).mean() < tol and np.abs(rgb[..., 1] - rgb[..., 2]).mean() < tol


def drop_grayscale(paths):
    keep = []
    for p in tqdm(paths, desc="Checking for grayscale images"):
        if not is_grayscale(load_rgb(p)):
            keep.append(p)
    return keep


if not DATA_DIR.is_dir():
    raise FileNotFoundError(
        f"Dataset folder '{DATA_DIR.resolve()}' not found.\n"
        "Create a folder named 'dataset' next to this notebook and put your colour images in it "
        "(either directly, or in dataset/train and dataset/test)."
    )

train_dir, test_dir = DATA_DIR / "train", DATA_DIR / "test"
if train_dir.is_dir() and test_dir.is_dir():
    train_paths, test_paths = list_images(train_dir), list_images(test_dir)
    print("Using pre-split dataset: dataset/train and dataset/test")
else:
    all_paths = list_images(DATA_DIR)
    rng = random.Random(SEED)
    rng.shuffle(all_paths)
    n_train = int(round(len(all_paths) * TRAIN_FRACTION))
    train_paths, test_paths = all_paths[:n_train], all_paths[n_train:]
    print(f"Using flat dataset folder, random {TRAIN_FRACTION:.0%} / {1 - TRAIN_FRACTION:.0%} split")

train_paths, test_paths = drop_grayscale(train_paths), drop_grayscale(test_paths)
print(f"Training images: {len(train_paths)} | Test images: {len(test_paths)}")
assert len(train_paths) >= 5 and len(test_paths) >= 1, "Not enough colour images found in the dataset folder."

## 4. Segmentation and feature extraction

* **SLIC** superpixels are computed on the **Y (luminance)** channel only, since at test time that is all we have.
* Each superpixel is described by the **magnitude of the 2D FFT** of the 10x10 patch around its centroid
  (log-compressed, then standardised before the SVR; the FFT is complex so we use its magnitude to get real-valued features).
* Targets are the **mean U** and **mean V** of the pixels in the superpixel.

In [ ]:
def segment(Y):
    """SLIC superpixels on the luminance channel. Returns a label map with labels 0..n-1."""
    try:
        labels = slic(Y, n_segments=N_SEGMENTS, compactness=COMPACTNESS,
                      channel_axis=None, start_label=1)
    except TypeError:                                   # older scikit-image
        labels = slic(Y, n_segments=N_SEGMENTS, compactness=COMPACTNESS,
                      multichannel=False, start_label=1)
    labels, _, _ = relabel_sequential(labels)           # make labels contiguous (1..n)
    return labels - 1


def extract_features(Y, labels):
    """FFT-magnitude of the PATCH x PATCH window around each superpixel centroid -> (n_segments, PATCH*PATCH)."""
    n = labels.max() + 1
    half = PATCH // 2
    Yp = np.pad(Y, half, mode="reflect")
    centroids = ndimage.center_of_mass(np.ones_like(labels, dtype=float), labels, index=np.arange(n))
    feats = np.empty((n, PATCH * PATCH))
    for i, (cy, cx) in enumerate(centroids):
        y, x = int(round(cy)), int(round(cx))
        patch = Yp[y:y + PATCH, x:x + PATCH]            # window rows y-half .. y+half-1 in original coordinates
        feats[i] = np.log1p(np.abs(np.fft.fft2(patch))).ravel()
    return feats


def segment_means(channel, labels):
    return ndimage.mean(channel, labels, index=np.arange(labels.max() + 1))


def process_image(path, keep_rgb=True):
    """Full preprocessing of one image: RGB -> YUV, SLIC, features, per-segment mean U/V."""
    rgb = load_rgb(path)
    yuv = color.rgb2yuv(rgb)
    Y = yuv[..., 0]
    labels = segment(Y)
    sample = dict(
        path=Path(path),
        Y=Y,
        labels=labels,
        feats=extract_features(Y, labels),
        uv=np.column_stack([segment_means(yuv[..., 1], labels), segment_means(yuv[..., 2], labels)]),
    )
    if keep_rgb:
        sample["rgb"] = rgb
    return sample


# Quick visual check of the segmentation on one training image
demo = process_image(train_paths[0])
print(f"{demo['labels'].max() + 1} superpixels, feature matrix shape: {demo['feats'].shape}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].imshow(demo["Y"], cmap="gray"); ax[0].set_title("Input: Y channel (grayscale)")
ax[1].imshow(mark_boundaries(np.dstack([demo["Y"]] * 3), demo["labels"], color=(1, 1, 0)))
ax[1].set_title("SLIC superpixels")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Split the training images into "fit" and "validation" parts (validation is only used for hyperparameter tuning)
rng = random.Random(SEED)
shuffled = list(train_paths)
rng.shuffle(shuffled)
n_val = max(1, int(round(len(shuffled) * VAL_FRACTION))) if RUN_TUNING and len(shuffled) >= 10 else 0
val_paths, fit_paths = shuffled[:n_val], shuffled[n_val:]
print(f"Fit images: {len(fit_paths)} | Validation images: {len(val_paths)}")

# Superpixel features / targets of the fit images (images are processed one by one to save memory)
X_fit, UV_fit = [], []
for p in tqdm(fit_paths, desc="Extracting training features"):
    s = process_image(p, keep_rgb=False)
    X_fit.append(s["feats"]); UV_fit.append(s["uv"])
X_fit, UV_fit = np.vstack(X_fit), np.vstack(UV_fit)

# Validation images are kept in full (they are also used to tune the ICM stage)
val_samples = [process_image(p) for p in tqdm(val_paths, desc="Processing validation images")]
print("Training superpixels:", X_fit.shape[0])

## 5. Support Vector Regression (local colour estimates)

Two RBF-kernel SVRs (default scikit-learn Gaussian kernel, as in the paper): one predicts **U**, the other **V**.
When `RUN_TUNING = True`, `C` and `epsilon` are chosen per channel on the validation images.

In [ ]:
scaler = StandardScaler().fit(X_fit)
Xs_fit = scaler.transform(X_fit)

best_params = {0: (SVR_C, SVR_EPSILON), 1: (SVR_C, SVR_EPSILON)}   # channel 0 = U, channel 1 = V

if RUN_TUNING and val_samples:
    X_val = scaler.transform(np.vstack([s["feats"] for s in val_samples]))
    UV_val = np.vstack([s["uv"] for s in val_samples])
    sub = np.random.RandomState(SEED).choice(len(Xs_fit), min(len(Xs_fit), TUNE_SUBSAMPLE), replace=False)

    for ch, name in enumerate("UV"):
        best_err = np.inf
        for C in C_GRID:
            for eps in EPS_GRID:
                m = SVR(kernel="rbf", gamma="scale", C=C, epsilon=eps).fit(Xs_fit[sub], UV_fit[sub, ch])
                err = mean_squared_error(UV_val[:, ch], m.predict(X_val))
                if err < best_err:
                    best_err, best_params[ch] = err, (C, eps)
        print(f"Channel {name}: best C={best_params[ch][0]}, epsilon={best_params[ch][1]}  (validation MSE {best_err:.5f})")
else:
    print("Tuning skipped - using C =", SVR_C, "epsilon =", SVR_EPSILON)

In [ ]:
# Final SVR training on ALL training images (fit + validation)
X_all = np.vstack([X_fit] + [s["feats"] for s in val_samples])
UV_all = np.vstack([UV_fit] + [s["uv"] for s in val_samples])
scaler = StandardScaler().fit(X_all)
Xs_all = scaler.transform(X_all)

if MAX_TRAIN_SAMPLES and len(Xs_all) > MAX_TRAIN_SAMPLES:
    idx = np.random.RandomState(SEED).choice(len(Xs_all), MAX_TRAIN_SAMPLES, replace=False)
    Xs_all, UV_all = Xs_all[idx], UV_all[idx]

print(f"Training the two SVRs on {len(Xs_all)} superpixels (this can take a few minutes) ...")
t0 = time.time()
svr_u = SVR(kernel="rbf", gamma="scale", C=best_params[0][0], epsilon=best_params[0][1]).fit(Xs_all, UV_all[:, 0])
svr_v = SVR(kernel="rbf", gamma="scale", C=best_params[1][0], epsilon=best_params[1][1]).fit(Xs_all, UV_all[:, 1])
print(f"Done in {time.time() - t0:.1f}s")

model = dict(scaler=scaler, svr_u=svr_u, svr_v=svr_v)

## 6. MRF smoothing with Iterated Conditional Modes (ICM)

For each channel the superpixels form an MRF. The energy of superpixel *i* with hidden chrominance *c_i* is

$$E_i = \frac{\lVert c_i - \mu_i \rVert_2^2}{2\sigma^2} + \gamma \sum_{x^{(j)} \in N(i)} \lVert c_i - c_j \rVert_2^2$$

where $\mu_i$ is the SVR prediction, $N(i)$ the adjacent superpixels, and an adjacent superpixel $j$ is only included if
$\lVert \phi(x^{(i)}) - \phi(x^{(j)}) \rVert_2$ is below a threshold (similar texture).
ICM updates one node at a time to its energy minimiser, which has a closed form here:

$$c_i \leftarrow \frac{\mu_i/\sigma^2 + 2\gamma \sum_{j \in N(i)} c_j}{1/\sigma^2 + 2\gamma\,|N(i)|}$$

and repeats until convergence.

In [ ]:
def build_neighbors(labels, feats_scaled, percentile=NEIGHBOR_PERCENTILE):
    """Adjacent superpixel pairs whose feature distance is below a per-image threshold."""
    pairs = np.concatenate([
        np.stack([labels[:, :-1].ravel(), labels[:, 1:].ravel()], 1),     # horizontal neighbours
        np.stack([labels[:-1, :].ravel(), labels[1:, :].ravel()], 1),     # vertical neighbours
    ])
    pairs = pairs[pairs[:, 0] != pairs[:, 1]]
    pairs = np.unique(np.sort(pairs, axis=1), axis=0)                      # undirected, unique edges
    dist = np.linalg.norm(feats_scaled[pairs[:, 0]] - feats_scaled[pairs[:, 1]], axis=1)
    pairs = pairs[dist <= np.percentile(dist, percentile)]

    neigh = [[] for _ in range(labels.max() + 1)]
    for a, b in pairs:
        neigh[a].append(b)
        neigh[b].append(a)
    return [np.array(n, dtype=int) for n in neigh]


def icm(mu, neigh, gamma=ICM_GAMMA, sigma=ICM_SIGMA, max_iter=ICM_MAX_ITER, tol=ICM_TOL):
    """Minimise the MRF energy of one channel with Iterated Conditional Modes."""
    c = mu.copy()
    inv_s2 = 1.0 / sigma ** 2
    for _ in range(max_iter):
        max_change = 0.0
        for i, nb in enumerate(neigh):
            if len(nb) == 0:
                continue
            new = (mu[i] * inv_s2 + 2 * gamma * c[nb].sum()) / (inv_s2 + 2 * gamma * len(nb))
            max_change = max(max_change, abs(new - c[i]))
            c[i] = new
        if max_change < tol:
            break
    return c


def predict_uv(model, feats):
    """SVR estimates of (U, V) for each superpixel -> (n, 2)."""
    X = model["scaler"].transform(feats)
    return np.column_stack([model["svr_u"].predict(X), model["svr_v"].predict(X)])


def render(Y, labels, uv):
    """Combine the original Y with per-superpixel U, V and convert back to RGB."""
    yuv = np.dstack([Y, uv[labels, 0], uv[labels, 1]])
    return np.clip(color.yuv2rgb(yuv), 0, 1)


def colorize_sample(model, s, gamma=ICM_GAMMA, sigma=ICM_SIGMA):
    """Returns (RGB from SVR only, RGB from SVR + ICM)."""
    mu = predict_uv(model, s["feats"])
    neigh = build_neighbors(s["labels"], model["scaler"].transform(s["feats"]))
    smooth = np.column_stack([icm(mu[:, c], neigh, gamma, sigma) for c in range(2)])
    return render(s["Y"], s["labels"], mu), render(s["Y"], s["labels"], smooth)


def rgb_errors(pred, orig):
    """(mean squared error, mean per-pixel Euclidean RGB distance), RGB scaled to [0, 1]."""
    diff = pred - orig
    return float(np.mean(diff ** 2)), float(np.mean(np.linalg.norm(diff, axis=-1)))

In [ ]:
# Tune the MRF parameters (gamma, sigma) on the validation images
if RUN_TUNING and val_samples:
    cache = []
    for s in val_samples:
        mu = predict_uv(model, s["feats"])
        neigh = build_neighbors(s["labels"], model["scaler"].transform(s["feats"]))
        cache.append((s, mu, neigh))

    svr_err = np.mean([rgb_errors(render(s["Y"], s["labels"], mu), s["rgb"])[0] for s, mu, _ in cache])
    best_icm, best_err = (ICM_GAMMA, ICM_SIGMA), np.inf
    for sigma in SIGMA_GRID:
        for gamma in GAMMA_GRID:
            errs = []
            for s, mu, neigh in cache:
                smooth = np.column_stack([icm(mu[:, c], neigh, gamma, sigma) for c in range(2)])
                errs.append(rgb_errors(render(s["Y"], s["labels"], smooth), s["rgb"])[0])
            e = float(np.mean(errs))
            if e < best_err:
                best_err, best_icm = e, (gamma, sigma)
    ICM_GAMMA, ICM_SIGMA = best_icm
    print(f"Validation RGB MSE  SVR only: {svr_err:.5f} | SVR+ICM: {best_err:.5f}")
    print(f"Chosen ICM parameters: gamma={ICM_GAMMA}, sigma={ICM_SIGMA}")
else:
    print(f"Using ICM parameters gamma={ICM_GAMMA}, sigma={ICM_SIGMA}")

joblib.dump(dict(model=model, gamma=ICM_GAMMA, sigma=ICM_SIGMA), MODEL_DIR / "colorizer.joblib")
print("Model saved to", MODEL_DIR / "colorizer.joblib")

## 7. Evaluation on the test set

Three stages are compared against the original colour image: the **grayscale input**, the **SVR-only** prediction and the **SVR + ICM** prediction.
Errors are computed on RGB values scaled to [0, 1]: mean squared error (MSE) and mean per-pixel Euclidean distance.

In [ ]:
records, gallery = [], []
(OUTPUT_DIR / "colorized").mkdir(parents=True, exist_ok=True)

for k, p in enumerate(tqdm(test_paths, desc="Colorizing test images")):
    s = process_image(p)
    rgb_svr, rgb_icm = colorize_sample(model, s, ICM_GAMMA, ICM_SIGMA)
    gray = np.dstack([s["Y"]] * 3)

    records.append([rgb_errors(gray, s["rgb"]), rgb_errors(rgb_svr, s["rgb"]), rgb_errors(rgb_icm, s["rgb"])])
    Image.fromarray((rgb_icm * 255).astype(np.uint8)).save(OUTPUT_DIR / "colorized" / f"{p.stem}_colorized.png")
    if k < N_SHOW:
        gallery.append((s["rgb"], gray, rgb_svr, rgb_icm))

records = np.array(records)                  # (n_images, 3 stages, 2 metrics)
mse = records[:, :, 0].mean(axis=0)
dist = records[:, :, 1].mean(axis=0)

print(f"{'Stage':<18}{'Avg MSE':>14}{'Avg RGB distance':>20}")
for name, m, d in zip(["Grayscale input", "SVR only", "SVR + ICM"], mse, dist):
    print(f"{name:<18}{m:>14.5f}{d:>20.5f}")

print(f"\nSVR reduced the MSE by {100 * (1 - mse[1] / mse[0]):.1f}% relative to grayscale")
print(f"ICM reduced it by a further {100 * (mse[1] - mse[2]) / mse[0]:.1f} percentage points")
print(f"(ICM improved {100 * np.mean(records[:, 2, 0] < records[:, 1, 0]):.0f}% of the test images)")

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 3.5))
bars = ax.bar(["Grayscale", "SVR only", "SVR + ICM"], mse, color=["#9e9e9e", "#4c8bd6", "#e07b39"])
ax.set_ylabel("Average RGB MSE (lower is better)")
ax.set_title("Error before / after each stage")
for b, v in zip(bars, mse):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.4f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "error_comparison.png", dpi=150)
plt.show()

In [ ]:
titles = ["Original", "Grayscale input", "SVR only", "SVR + ICM (final)"]
fig, axes = plt.subplots(len(gallery), 4, figsize=(14, 3.1 * len(gallery)), squeeze=False)
for r, imgs in enumerate(gallery):
    for c, img in enumerate(imgs):
        axes[r, c].imshow(img)
        axes[r, c].axis("off")
        if r == 0:
            axes[r, c].set_title(titles[c], fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "results_gallery.png", dpi=150)
plt.show()

## 8. Colorize your own images

Put any grayscale (or colour) images in an **`input_images/`** folder next to the notebook and run the cell below.
Colour images are converted to grayscale internally and then colorized. Results are saved in `outputs/custom/`.
You can also reload the saved model later without retraining:
`saved = joblib.load("models/colorizer.joblib"); model = saved["model"]`.

In [ ]:
def colorize_file(path, model, gamma=ICM_GAMMA, sigma=ICM_SIGMA):
    s = process_image(path)
    _, rgb_icm = colorize_sample(model, s, gamma, sigma)
    return np.dstack([s["Y"]] * 3), rgb_icm


custom_paths = list_images(INPUT_DIR) if INPUT_DIR.is_dir() else []
if not custom_paths:
    print(f"No images found in '{INPUT_DIR}/'. Create that folder and add images to try your own.")
else:
    (OUTPUT_DIR / "custom").mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(len(custom_paths), 2, figsize=(9, 3.4 * len(custom_paths)), squeeze=False)
    for r, p in enumerate(custom_paths):
        gray, colorized = colorize_file(p, model)
        Image.fromarray((colorized * 255).astype(np.uint8)).save(OUTPUT_DIR / "custom" / f"{p.stem}_colorized.png")
        axes[r, 0].imshow(gray); axes[r, 0].set_title(f"{p.name} (input)"); axes[r, 0].axis("off")
        axes[r, 1].imshow(colorized); axes[r, 1].set_title("Colorized"); axes[r, 1].axis("off")
    plt.tight_layout(); plt.show()

## 9. Notes and known limitations (from the paper)

* The model only works well on images **similar to the training set** (same theme).
* Colours under-represented in the training data (e.g. browns / yellows if the data is mostly blue / green) are reproduced poorly.
* Colours may "bleed" between neighbouring superpixels because the 10x10 patch around a small superpixel covers its neighbours.
* Outputs tend to be **desaturated**, because an SVR predicts one value (effectively an average of plausible colours).
  A multimodal / distribution-based predictor would help.
* Possible extensions: CIELAB colour space, richer features (variance, luminance histogram), a pixel-wise MRF, CNN-based features.

**Reference:** H. Ho and V. Ramesh, *Automatic Image Colorization*, Stanford CS229. Original code: https://github.com/harrison8989/recolorizer